In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Data/b2busers.csv')
df

,customerID,month,logins,churned
0,1,-12,106,True
1,1,-11,118,True
2,1,-10,117,True
3,1,-9,127,True
4,1,-8,102,True
...,...,...,...,...
2395,200,-5,61,False
2396,200,-4,57,False
2397,200,-3,55,False
2398,200,-2,63,False


## Calcolo del segnale 
Da numerose osservazioni per cliente vogliamo avere un solo numero per cliente, il suo segnale. Che risponde alla domanda “ sta usando il prodotto meno di prima?” Credo valga anche Come sta usando il prodotto.

In [3]:
#filtro i mesi -12to-7,raggruppo per customer, media degli accessi
df_riferimento = df[df["month"].between(-12, -7)].groupby(['customerID', 'churned'], as_index=False)['logins'].mean()
df_riferimento = df_riferimento.rename(columns={"logins": "riferimento_mean"})

#filtro i mesi -6to-3,raggruppo per customer, media degli accessi
df_osservazione = df[df["month"].between(-6, -3)].groupby(['customerID', 'churned'], as_index=False)['logins'].mean()
df_osservazione = df_osservazione.rename(columns={"logins": "osservazione_mean"})

#unisco in un df
df_unito = df_riferimento.merge(df_osservazione, on="customerID")
df_unito = df_unito.rename(columns={"churned_y": "churned"})

#calcolo variazione di accessi per customer: (obs-reference)/reference * 100
df_unito['variazione'] = round((df_unito['osservazione_mean'] - df_unito['riferimento_mean'])/df_unito['riferimento_mean'] * 100,2)

df_unito = df_unito[['customerID', 'riferimento_mean','osservazione_mean', 'variazione', 'churned']]

df_unito

,customerID,riferimento_mean,osservazione_mean,variazione,churned
0,1,113.500000,81.75,-27.97,True
1,2,90.333333,57.75,-36.07,True
2,3,122.000000,87.50,-28.28,True
3,4,183.500000,137.25,-25.20,True
4,5,14.000000,9.00,-35.71,True
...,...,...,...,...,...
195,196,95.000000,94.75,-0.26,False
196,197,147.166667,145.75,-0.96,False
197,198,197.833333,196.00,-0.93,False
198,199,175.000000,180.50,3.14,False


## Assegno score 
converto il segnale da valore continuo a uno score. Decido tramite deviazione standard la soglia che divide gli score

In [4]:
df_unito["risk"] = np.where(df_unito["variazione"] <= -20, "high", "low")

In [11]:
print('% di churn nel gruppo High', df_unito[df_unito['risk'] == 'high']['churned'].value_counts())
print('% di churn nel gruppo Low', df_unito[df_unito['risk'] == 'low']['churned'].value_counts())

% di churn nel gruppo High churned
False    13
True     10
Name: count, dtype: int64
% di churn nel gruppo Low churned
False    157
True      20
Name: count, dtype: int64


## Considerazioni
Nel mio dataset artificiale ho utenti:
- 15    calo,       churned
- 20    calo,       no churned (falsi posiitiv)
- 15    no calo,    churned
- 150   no calo,    no churned

Clienti High: 23, di cui churned 10, quindi 43%<br>
Clienti Low: 177, di cui churned 20, quindi 11%<br>
Lif: clienti High hanno 3,8 le probabilità di churnare.

Ho intercettato 10 churned su 30.
IL segnale di utilizzo è utile ma non sufficiente, gli altri 20 possono essere identificato da altri segnali.

